# Sesión 3 — Práctica: Agente LangGraph para vuestro Negocio

Esta práctica tiene **dos partes**:

**Parte 1 — Técnica (individual, ~20 min)**  
Completad el agente de atención al cliente implementando los nodos que faltan.

**Parte 2 — Estratégica (grupal, ~20 min)**  
Diseñad (y si os da tiempo, implementad) un agente LangGraph para un proceso de vuestro sector.

---

> Las celdas marcadas con `# TODO` son las que debéis completar.

## Setup

In [ ]:
%pip install -q \
    langgraph \
    langchain \
    langchain-google-genai \
    langchain-ollama \
    langchain-chroma \
    chromadb \
    "datasets<3.0" \
    pandas

In [ ]:
# ── Configuración ────────────────────────────────────────────────────────────
BACKEND = "gemini"          # "gemini" | "ollama"
OLLAMA_MODEL = "qwen3:4b"
OLLAMA_EMBEDDING_MODEL = "nomic-embed-text-v2-moe"
# ─────────────────────────────────────────────────────────────────────────────

if BACKEND == "gemini":
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

    from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=GOOGLE_API_KEY
    )
    embeddings = GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001",
        google_api_key=GOOGLE_API_KEY
    )
    print("✓ Backend: Gemini 2.5 Flash + gemini-embedding-001")

elif BACKEND == "ollama":
    from langchain_ollama import ChatOllama, OllamaEmbeddings
    llm = ChatOllama(model=OLLAMA_MODEL)
    embeddings = OllamaEmbeddings(model=OLLAMA_EMBEDDING_MODEL)
    print(f"✓ Backend: Ollama — {OLLAMA_MODEL} + {OLLAMA_EMBEDDING_MODEL}")

else:
    raise ValueError(f"Backend desconocido: {BACKEND}")

In [ ]:
# Vector store — memoria de largo plazo del agente (igual que en la Sesión 2)
from datasets import load_dataset
import pandas as pd
from langchain_core.documents import Document
from langchain_chroma import Chroma

print("Construyendo la memoria de largo plazo...")
ds = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Electronics",
    split="full",
    streaming=True,
    trust_remote_code=True,
)
df = pd.DataFrame(ds.take(200))[["title", "text", "rating", "parent_asin"]].dropna(subset=["text"])
df["rating"] = df["rating"].astype(int)

docs = [
    Document(
        page_content=f"{row['title']}\n\n{row['text']}",
        metadata={"rating": row["rating"], "parent_asin": row["parent_asin"]}
    )
    for _, row in df.iterrows()
]

vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"✓ {vectorstore._collection.count()} documentos indexados")

---

## Parte 1: Completad el Agente

El estado y algunos nodos ya están implementados. Vuestro trabajo es completar los que faltan y cablear el grafo.

In [ ]:
from typing import TypedDict

class TicketState(TypedDict):
    ticket: str
    category: str
    urgency: str
    summary: str
    rag_context: str
    draft_response: str
    escalate: bool
    final_response: str

### Ejercicio 1: Nodo `clasificar`

Implementad el nodo de clasificación. Debe llamar al LLM y devolver un dict con:
- `category`: una de `["devolucion", "consulta_tecnica", "queja", "envio", "otro"]`
- `urgency`: una de `["alta", "media", "baja"]`
- `summary`: resumen del ticket en máximo 15 palabras

El LLM debe responder en JSON puro (sin markdown). Manejad el caso en que el LLM añada bloques de código.

In [ ]:
import json

# TODO: implementa el nodo clasificar
def clasificar(state: TicketState) -> dict:
    pass

# Prueba rápida
test_state: TicketState = {
    "ticket": "Mi auricular derecho no funciona desde ayer. Quiero devolverlo.",
    "category": "", "urgency": "", "summary": "",
    "rag_context": "", "draft_response": "",
    "escalate": False, "final_response": ""
}

resultado = clasificar(test_state)
print(resultado)

### Nodo `consultar_catalogo` (ya implementado)

Este nodo usa el retriever para recuperar reviews relevantes. Observad cómo construye la consulta combinando categoría y resumen.

In [ ]:
def consultar_catalogo(state: TicketState) -> dict:
    consulta = f"{state.get('category', '')} {state.get('summary', state['ticket'])}"
    docs = retriever.invoke(consulta)
    contexto = "\n\n---\n\n".join(
        f"[{d.metadata.get('rating', '?')}★] {d.page_content[:350]}"
        for d in docs
    )
    print(f"  [consultar_catalogo] → {len(docs)} reviews recuperadas")
    return {"rag_context": contexto}

print("✓ Nodo 'consultar_catalogo' listo")

### Ejercicio 2: Nodo `redactar_respuesta`

Implementad el nodo que genera el borrador de respuesta.

El prompt debe:
- Dar un rol al LLM (responsable de atención al cliente)
- Incluir el ticket original, la categoría y el contexto del catálogo
- Pedir una respuesta de máximo 3 frases
- Devolver `{"draft_response": ...}`

In [ ]:
# TODO: implementa el nodo redactar_respuesta
def redactar_respuesta(state: TicketState) -> dict:
    pass

print("✓ Nodo 'redactar_respuesta' definido" if redactar_respuesta else "⚠ Pendiente")

### Nodos de decisión (ya implementados)

In [ ]:
def evaluar_escalado(state: TicketState) -> dict:
    categorias_criticas = {"devolucion", "queja"}
    escalar = (
        state.get("urgency") == "alta"
        and state.get("category") in categorias_criticas
    )
    print(f"  [evaluar_escalado] → escalar: {escalar}")
    return {"escalate": escalar}

def respuesta_automatica(state: TicketState) -> dict:
    return {"final_response": f"[AUTOMÁTICO]\n\n{state['draft_response']}"}

def escalar_a_humano(state: TicketState) -> dict:
    return {"final_response": (
        f"[ESCALADO]\nCategoría: {state.get('category')} | Urgencia: {state.get('urgency')}\n"
        f"Resumen: {state.get('summary')}\n\nBorrador:\n{state['draft_response']}"
    )}

def decidir_ruta(state: TicketState) -> str:
    return "escalar" if state.get("escalate") else "enviar"

print("✓ Nodos de decisión listos")

### Ejercicio 3: Cablear el grafo

Construid el grafo LangGraph conectando todos los nodos.

Flujo esperado:
```
START → clasificar → consultar_catalogo → redactar_respuesta → evaluar_escalado
                                                                      ↓
                                              escalar_a_humano ←── ¿escalar?
                                              respuesta_automatica ←── ¿enviar?
                                                      ↓                ↓
                                                     END              END
```

In [ ]:
from langgraph.graph import StateGraph, END

# TODO: construye y compila el grafo
builder = StateGraph(TicketState)

# TODO: add_node para cada función

# TODO: set_entry_point

# TODO: add_edge para las transiciones fijas

# TODO: add_conditional_edges para el nodo de escalado

# TODO: add_edge hacia END para los dos nodos finales

agent = None  # reemplazar con builder.compile()

if agent:
    print("✓ Agente compilado")
    try:
        from IPython.display import display, Image
        display(Image(agent.get_graph().draw_mermaid_png()))
    except Exception:
        print(agent.get_graph().draw_mermaid())

### Ejercicio 4: Probar el agente

Probad vuestro agente con al menos dos tickets. Verificad que:
- Los tickets de devolución urgente se escalan
- Las consultas técnicas se responden automáticamente

In [ ]:
tickets_prueba = [
    "Compré unos auriculares hace dos semanas y están completamente rotos. Quiero mi dinero de vuelta YA.",
    "¿El altavoz Bluetooth modelo X es compatible con iOS 17?",
]

for ticket in tickets_prueba:
    if agent is None:
        print("⚠ Primero completad el Ejercicio 3 para tener el agente compilado.")
        break

    print(f"\n{'='*60}")
    print(f"TICKET: {ticket}")
    print("="*60)

    estado_inicial: TicketState = {
        "ticket": ticket, "category": "", "urgency": "", "summary": "",
        "rag_context": "", "draft_response": "",
        "escalate": False, "final_response": ""
    }
    resultado = agent.invoke(estado_inicial)
    print(f"\n{resultado['final_response']}")

### Ejercicio 5 (bonus): Modificad la lógica de escalado

La función `evaluar_escalado` actual usa una regla fija. Modificadla para que también escale cuando el ticket menciona explícitamente palabras como "abogado", "denuncia" o "consumidor".

Pista: podéis usar detección de palabras clave sobre `state['ticket']`, o pedirle al LLM que evalúe el nivel de riesgo legal.

In [ ]:
# TODO (bonus): evaluar_escalado mejorado con detección de riesgo legal
def evaluar_escalado_v2(state: TicketState) -> dict:
    pass

---

## Parte 2: Diseñad un Agente para vuestro Negocio

**Grupos de 3–4 personas. 20 minutos.**

Elegid un proceso de vuestro sector y diseñad un agente LangGraph.  
**Presentación**: 3 minutos por grupo.

### El proceso elegido

**Nombre del proceso:**  
*[Completad aquí]*

**Sector / empresa:**  
*[Completad aquí]*

**Problema que resuelve el agente:**  
*[Completad aquí]*

### Estado del agente

¿Qué información fluye por el grafo? Definid el TypedDict:

```python
class MiAgenteState(TypedDict):
    # TODO: definid los campos
    pass
```

*[Completad con vuestro diseño]*

### Nodos del grafo

| Nodo | Qué hace | ¿LLM, regla o API? | Inputs del estado | Outputs al estado |
|------|----------|-------------------|-------------------|-------------------|
| *[nodo 1]* | | | | |
| *[nodo 2]* | | | | |
| *[nodo 3]* | | | | |
| *[nodo 4]* | | | | |

### Flujo y condiciones

**¿Hay aristas condicionales? ¿Cuándo se bifurca el grafo?**

*[Completad aquí — podéis usar texto o un diagrama ASCII]*

---

**¿Cuándo escala a un humano? ¿Por qué?**

*[Completad aquí]*

---

**¿Qué memoria necesita el agente?**  
*(¿Vector store de documentos propios? ¿Historial de conversaciones? ¿Base de datos externa?)*

*[Completad aquí]*

### Gobernanza

**¿Qué acciones con efectos reales puede tomar el agente?**  
*(enviar emails, modificar registros, hacer pagos...)*

*[Completad aquí]*

---

**¿Qué límites de acción definiríais?**  
*(máximo N acciones/hora, importe máximo, tipo de datos que no puede leer...)*

*[Completad aquí]*

---

**¿Qué loggaríais para poder auditar el agente?**

*[Completad aquí]*

In [ ]:
# Bonus: implementad el grafo de vuestro agente aquí
# No es obligatorio — si lo implementáis, probadlo con un caso de prueba real

from langgraph.graph import StateGraph, END
from typing import TypedDict

# TODO (bonus): vuestro agente
pass